# MedAgent: 02. ECG Preprocessing, AAMI Beat Segmentation & DataLoaders
## Automated Heartbeat Extraction and PyTorch Dataset Pipeline

This notebook implements the AAMI EC57 standard heartbeat segmentation:
1. Mapping PhysioNet annotation symbols to **5 AAMI classes** ($N, S, V, F, Q$).
2. Windowing around R-peaks: $260$ samples ($90$ pre-peak, $170$ post-peak $\approx 722\text{ ms}$).
3. Extracting and analyzing `data/processed/signals/mitbih_aami_beats.npz`.
4. Plotting average heartbeat morphology per arrhythmia category.
5. Signal augmentation (Gaussian noise, amplitude scaling, time shifting).
6. Building PyTorch `ECGDataset` and stratified `DataLoader`s with class weights.

In [ ]:
import sys
from pathlib import Path

# Robust project root determination whether cwd is MedAgent or MedAgent/notebooks
current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure venv site-packages are accessible even if run from a generic kernel
VENV_SITE = PROJECT_ROOT / "venv" / "Lib" / "site-packages"
if VENV_SITE.exists() and str(VENV_SITE) not in sys.path:
    sys.path.insert(0, str(VENV_SITE))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data.preprocessing.ecg_cleaner import (
    AAMI_CLASSES,
    AAMI_SYMBOL_MAP,
    extract_beats_from_record,
    segment_beats_from_r_peaks,
    clean_ecg
)
from src.data.loaders.ecg_dataset import (
    ECGAugmentation,
    ECGDataset,
    get_ecg_dataloaders
)

print("ECG Preprocessing modules loaded successfully!")

### 1. AAMI EC57 Arrhythmia Classes
| Class ID | Class Name | Included PhysioNet Symbols |
|:---|:---|:---|
| **0** | **Normal (N)** | Normal (`N`), Left bundle branch block (`L`), Right bundle branch block (`R`), Atrial escape (`e`), Nodal escape (`j`) |
| **1** | **Supraventricular (S)** | Atrial premature (`A`), Aberrated atrial premature (`a`), Nodal premature (`J`), Supraventricular premature (`S`) |
| **2** | **Ventricular (V)** | Premature ventricular contraction (`V`), Ventricular escape (`E`) |
| **3** | **Fusion (F)** | Fusion of ventricular and normal beat (`F`) |
| **4** | **Paced / Unknown (Q)** | Paced beat (`/`), Fusion of paced beat (`f`), Unclassifiable (`Q`) |

In [ ]:
print("AAMI Classes Defined:")
for cid, name in AAMI_CLASSES.items():
    print(f"  Class {cid}: {name}")

### 2. Inspect Preprocessed Dataset (`mitbih_aami_beats.npz`)

In [ ]:
npz_path = PROJECT_ROOT / "data" / "processed" / "signals" / "mitbih_aami_beats.npz"
data = np.load(str(npz_path))

beats = data["beats"]
labels = data["labels"]
record_origins = data["record_origins"] if "record_origins" in data else np.array(["unknown"] * len(labels))

print(f"Total Segmented Beats: {len(beats)}")
print(f"Beat Window Shape: {beats.shape} (N_samples, Window_length)")
print(f"Unique Labels: {np.unique(labels)}")

# Class distribution
unique_classes, counts = np.unique(labels, return_counts=True)
plt.figure(figsize=(10, 4))
bars = plt.bar([AAMI_CLASSES[c] for c in unique_classes], counts, color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"])
plt.title("AAMI Arrhythmia Class Distribution in Dataset", fontsize=13, fontweight="bold")
plt.xlabel("Arrhythmia Category")
plt.ylabel("Number of Heartbeats")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f"{count:,}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

### 3. Morphology Analysis: Mean Waveforms per Arrhythmia Class

In [ ]:
time_ms = (np.arange(260) - 90) * (1000.0 / 360.0)  # Center 0 at R-peak

fig, axes = plt.subplots(1, len(unique_classes), figsize=(16, 4), sharey=True)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for i, cls_id in enumerate(unique_classes):
    cls_beats = beats[labels == cls_id]
    mean_beat = np.mean(cls_beats, axis=0)
    std_beat = np.std(cls_beats, axis=0)
    
    ax = axes[i]
    ax.plot(time_ms, mean_beat, color=colors[cls_id % len(colors)], lw=2, label="Mean Beat")
    ax.fill_between(time_ms, mean_beat - std_beat, mean_beat + std_beat, color=colors[cls_id % len(colors)], alpha=0.25, label="±1 STD")
    ax.axvline(0, color="gray", linestyle=":", alpha=0.8, label="R-Peak (t=0)")
    ax.set_title(AAMI_CLASSES[cls_id], fontsize=11, fontweight="bold")
    ax.set_xlabel("Time from R-Peak (ms)")
    if i == 0:
        ax.set_ylabel("Normalized Amplitude")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### 4. Data Augmentation Verification

In [ ]:
sample_beat = beats[0]
aug = ECGAugmentation(noise_std=0.04, scale_min=0.85, scale_max=1.15, shift_max_samples=6)

plt.figure(figsize=(12, 4))
plt.plot(time_ms, sample_beat, "k-", lw=2, label="Original Beat")
for i in range(3):
    aug_sample = aug(sample_beat)
    plt.plot(time_ms, aug_sample, linestyle="--", lw=1.2, alpha=0.8, label=f"Augmentation #{i+1}")

plt.title("ECG Beat Data Augmentation (Noise + Scaling + Jitter)", fontsize=13, fontweight="bold")
plt.xlabel("Time from R-Peak (ms)")
plt.ylabel("Normalized Amplitude")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 5. PyTorch DataLoader Integration & Batch Inspection

In [ ]:
train_loader, val_loader, test_loader, class_weights = get_ecg_dataloaders(
    npz_path=npz_path,
    batch_size=64,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    augment_train=True
)

print(f"Train Batches: {len(train_loader)} ({len(train_loader.dataset)} beats)")
print(f"Val Batches:   {len(val_loader)} ({len(val_loader.dataset)} beats)")
print(f"Test Batches:  {len(test_loader)} ({len(test_loader.dataset)} beats)")
print(f"Class Weights Tensor: {class_weights}")

# Inspect first batch
batch_x, batch_y = next(iter(train_loader))
print(f"Batch Tensor Shape: {batch_x.shape} (Batch, Channels, Sequence)")
print(f"Batch Labels Shape: {batch_y.shape}")